In [ ]:
%pip install sentence_transformers matplotlib nltk statsmodels accelerate seaborn

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
import matplotlib.pyplot as plt
import re
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize, word_tokenize
import numpy as np
from time import time
from huggingface_hub import login
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import statsmodels.api as sm
import ast
import os

## partie 0 : DATA & MODEL

In [ ]:
chemin_fichier = "donnees/FOMC_statements_characteristics_050525.csv"
df=pd.read_csv(chemin_fichier)

on part de 142972 statements et 358 meetings

In [ ]:
df["Statement"] = df["Statement"].fillna("").astype(str)

#on enlève les @action car ils ne sont pas bien pris en compte par le modele
df["Statement"] = df["Statement"].apply(lambda x: re.sub(r"@\w+", "", x).strip())
df["Statement"] = df["Statement"].apply(lambda x: re.sub(r"\[.*?\]", "", x).strip())

#on garde que les membres de la FOCM, cela revient à exclure le staff
df = df[df["Member"] == "M"].copy()

In [ ]:
df_speaker = df[['Date', 'Speaker', 'Statement', 'Member']].copy()

In [ ]:
##si on veut travailler avec SBERT
#model = SentenceTransformer('all-MiniLM-L6-v2')

##si on veut travailler avec SBERT finetuné
model = SentenceTransformer("finetuned-minilm-similarity-epoch5")

##si on veut travailler avec GTE
#model = SentenceTransformer("thenlper/gte-small")

## partie 1 : EMBEDDING ET MESURES

### etape1 : on réduit les prises de paroles à une seule par personne, la plus longue

In [ ]:
#colonne contenant la longueur du statement
df_speaker['Statement_length'] = df_speaker['Statement'].str.len()

#pour chaque individu on garde seulement sa déclaration la plus longue
df_speaker = df_speaker.loc[df_speaker.groupby(['Date', 'Speaker'])['Statement_length'].idxmax()].reset_index(drop=True)

minilm_tokenizer = SentenceTransformer('all-MiniLM-L6-v2')._first_module().tokenizer
df_speaker['token_count'] = df_speaker['Statement'].apply(lambda x: len(minilm_tokenizer.tokenize(x)))

In [ ]:
#pour l'importation sur git, on peut ne pas l'utiliser et directement charger le token avec login("hf_...")
from dotenv import load_dotenv
load_dotenv()  # charge .env

In [ ]:
#pour se connecter à hugging face et importer mistral
#si le token n'est plus valide il faut en créer un autre et copier coller le lien dans le login
login(token=os.getenv("HF_TOKEN"))


In [ ]:
#on utilise MISTRAL-7B-Instruct-v0.2
#il y'a une v0.3 disponible mais non compatible avec l'environnement python actuel

summarizer = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device_map={"": 0},
    torch_dtype=torch.float16
)

def summarize_statement(text):
    token_count = len(minilm_tokenizer.tokenize(text))
    
    # Si déjà ≤ 512 tokens, on garde tel quel
    if token_count <= 512:
        return text

    # Sinon, résumé avec prompt instruct
    prompt = f"<s>[INST] Reformulate the following text as if it had been spoken by a person during a meeting. Keep the output under 512 tokens : {text} [/INST]"

    try:
        output = summarizer(prompt, max_new_tokens=512, do_sample=False)[0]["generated_text"]
        summary = output.split("[/INST]")[-1].strip()
        summary_token_count = len(minilm_tokenizer.tokenize(summary))

        if summary_token_count > 512:
            summary = " ".join(summary.split()[:512])
        
        return summary
    
    #on interrompt pas le process, il y'aura à la place du résumé écrit [ERROR]
    #ça arrive avec onyxia si la phrase à résumer est trop longue car la mémoire est déjà quasi saturée avec le téléchargement de mistral
    except Exception as e:
        print(f"Error summarizing: {e}")
        return "[ERROR]"

In [ ]:
# Application à df_speaker avec barre de progression
df_finalred = df_speaker.copy()
df_finalred["Statement"] = [
    summarize_statement(text) for text in tqdm(df_finalred["Statement"], desc="Summarizing")
]

# Sauvegarde
#df_finalred.to_csv("df_finalred.csv", index=False)
#print("Résumés sauvegardés dans df_finalred.csv")

### etape2 : on calcule l'embedding de chacune des phrases

In [ ]:
chemin_fichier = "donnees/df_ENTIERrésuméMistral.csv"
df_finalred=pd.read_csv(chemin_fichier)
df_finalred=df_finalred[['Date','Speaker','Statement']]
#df_finalred['tokenred_count'] = df_finalred['Statement'].apply(lambda x: len(minilm_tokenizer.tokenize(x)))
df_finalred

In [ ]:
tqdm.pandas(desc="Computing embeddings")
df_finalred["embedding"] = df_finalred["Statement"].progress_apply(lambda x: model.encode(x, show_progress_bar=False).tolist())

In [ ]:
def cosine_sim(vec1, vec2):
    return cosine_similarity([vec1], [vec2])[0][0]

In [ ]:
def compute_influence_from_df(df_speaker):
    import numpy as np

    df = df_speaker.copy()
    df["influence"] = np.nan
    df["fs"] = np.nan
    df["bs"] = np.nan

    # Récupérer les dates dans l'ordre chronologique
    unique_dates = sorted(df["Date"].unique())

    # S'assurer qu'il y a au moins 3 dates pour pouvoir calculer quelque chose
    if len(unique_dates) < 3:
        raise ValueError("Il doit y avoir au moins 3 dates distinctes dans df_speaker")

    # Parcourir toutes les dates sauf la première et la dernière
    for i in range(1, len(unique_dates) - 1):
        date_before = unique_dates[i - 1]
        date_middle = unique_dates[i]
        date_after = unique_dates[i + 1]

        df_before = df[df["Date"] == date_before]
        df_middle = df[df["Date"] == date_middle]
        df_after = df[df["Date"] == date_after]

        emb_before = list(df_before["embedding"])
        emb_after = list(df_after["embedding"])

        for idx, row in df_middle.iterrows():
            vec = row["embedding"]

            fs = np.mean([cosine_sim(vec, ea) for ea in emb_after])
            bs = np.mean([cosine_sim(vec, eb) for eb in emb_before])

            influence = fs / bs if bs != 0 else float("inf")

            df.at[idx, "fs"] = fs
            df.at[idx, "bs"] = bs
            df.at[idx, "influence"] = influence

    return df

In [ ]:
df_compare=compute_influence_from_df(df_finalred)

### etape3 : on obtient les résultats

In [ ]:
##SANS FILTRAGE
# Recalcul propre sans utiliser la colonne "influence" du DataFrame initial
df_mean_scores = (
    df_compare
    .dropna(subset=["fs", "bs"])
    .groupby("Speaker", as_index=False)[["fs", "bs"]]
    .mean()
    .rename(columns={"fs": "FS", "bs": "BS"})
)

# Recalcul de l'influence après la moyenne
df_mean_scores["Influence"] = df_mean_scores["FS"] / df_mean_scores["BS"]

# Tri pour top 12 influence
top_influence = (
    df_mean_scores.sort_values(by="Influence", ascending=False)
    .head(12)
    .reset_index(drop=True)
    .rename(columns={"Speaker": "Top Influence Speaker"})
)

# Tri pour top 12 FS
top_fs = (
    df_mean_scores.sort_values(by="FS", ascending=False)
    .head(12)
    .reset_index(drop=True)
    .rename(columns={"Speaker": "Top FS Speaker"})
)

# Tri pour top 12 BS
top_bs = (
    df_mean_scores.sort_values(by="BS", ascending=False)
    .head(12)
    .reset_index(drop=True)
    .rename(columns={"Speaker": "Top BS Speaker"})
)

# Concaténation en tableau final
top_combined = pd.concat([top_influence, top_fs, top_bs], axis=1)
top_combined


In [ ]:
##AVEC FILTRAGE
# Nettoyage : s'assurer que les colonnes existent et sont valides
df_filtered = df_compare.copy()
df_filtered["Date"] = pd.to_datetime(df_filtered["Date"]).dt.year
df_filtered = df_filtered[df_filtered["Date"] >= 1979]

# Filtrer uniquement les speakers ayant suffisamment d'interventions
speaker_counts = df_filtered["Speaker"].value_counts()
eligible_speakers = speaker_counts[speaker_counts >= 12].index
df_filtered = df_filtered[df_filtered["Speaker"].isin(eligible_speakers)]

# Agrégation : moyenne de fs et bs par speaker
grouped = df_filtered.dropna(subset=["fs", "bs"]).groupby("Speaker")
df_mean_scores = grouped.agg({
    "fs": "mean",
    "bs": "mean"
}).reset_index()

# Recalculer correctement l'influence
df_mean_scores["Influence"] = df_mean_scores["fs"] / df_mean_scores["bs"]

# Renommer les colonnes pour affichage
df_mean_scores.rename(columns={"Speaker": "Top Influence Speaker", "fs": "FS", "bs": "BS"}, inplace=True)

# Générer les tops
top_influence = df_mean_scores.sort_values(by="Influence", ascending=False).head(12).reset_index(drop=True)
top_fs = df_mean_scores.sort_values(by="FS", ascending=False).head(12).reset_index(drop=True).rename(columns={"Top Influence Speaker": "Top FS Speaker"})
top_bs = df_mean_scores.sort_values(by="BS", ascending=False).head(12).reset_index(drop=True).rename(columns={"Top Influence Speaker": "Top BS Speaker"})

# Concaténer les résultats
top_combined = pd.concat([top_influence, top_fs, top_bs], axis=1)
top_combined

## partie 2 : INFLUENCE SELON EXPERIENCE

In [ ]:
def compute_rho_i(row, df):
    same_day_others = df[(df['Date'] == row['Date']) & (df['Speaker'] != row['Speaker'])]
    if same_day_others.empty:
        return np.nan
    sims = [cosine_sim(row['embedding'], other_emb) for other_emb in same_day_others['embedding']]
    return np.mean(sims)

In [ ]:
chemin_econo=("df_econometrie.csv")
df=pd.read_csv(chemin_econo)
df['embedding'] = df['embedding'].apply(ast.literal_eval)
df

In [ ]:
stats = pd.read_stata("22juillet.dta")
stats = stats[['Date', 'Speaker', 'vote_dissent', 'q_importance', 'xp', 'bankpres', 'Phd', 'gender']]
stats = stats.sort_values(by='Date').reset_index(drop=True)

In [ ]:
df = df.merge(stats[['Date', 'Speaker']], on=['Date', 'Speaker'], how='inner')
df = df.merge(stats[['Date', 'Speaker', 'vote_dissent', 'xp', 'bankpres', 'Phd', 'gender']], on=['Date', 'Speaker'], how='left')

In [ ]:
df

In [ ]:
import seaborn as sns

df = df[(df['influence'] >= -5) & (df['influence'] <= 5)]

# Filtrer les données valides (pas de NaN dans xp et influence)
df_plot = df[['xp', 'influence']].dropna()

# Taille et style du graphique
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_plot, x='xp', y='influence', alpha=0.5)

# Ajout d'une régression linéaire lissée pour voir la tendance (optionnel)
sns.regplot(data=df_plot, x='xp', y='influence', scatter=False, color='red', line_kws={'label':"Tendance (linéaire)"})

# Mise en forme
plt.title("Relation entre l'expérience (xp) et l'influence")
plt.xlabel("xp (années d'expérience)")
plt.ylabel("influence")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
df['influence'].describe()

In [ ]:
vars_to_use = ['influence', 'xp', 'Phd', 'gender', 'bankpres']
df_reg_clean = df[vars_to_use].dropna()

X = df_reg_clean[['xp', 'Phd', 'gender', 'bankpres']]
X = sm.add_constant(X)  # Ajoute l'ordonnée à l'origine (constante)
y = df_reg_clean['influence']

model = sm.OLS(y, X).fit()
model.summary()

## partie 3 : MESURE DE DIVERSITE

In [ ]:
chemin_econo=("df_econometrie.csv")
df=pd.read_csv(chemin_econo)
df['embedding'] = df['embedding'].apply(ast.literal_eval)
df

In [ ]:
stats = pd.read_stata("22juillet.dta")
stats = stats[['Date', 'Speaker', 'vote_dissent', 'q_importance', 'xp', 'bankpres', 'Phd', 'gender']]
stats = stats.sort_values(by='Date').reset_index(drop=True)
stats

In [ ]:
df = df.merge(stats[['Date', 'Speaker']], on=['Date', 'Speaker'], how='inner')
df = df.merge(stats[['Date', 'Speaker', 'vote_dissent', 'q_importance', 'xp', 'bankpres', 'Phd', 'gender']], on=['Date', 'Speaker'], how='left')

In [ ]:
df

In [ ]:
def compute_rho_i(row, df):
    same_day_others = df[(df['Date'] == row['Date']) & (df['Speaker'] != row['Speaker'])]
    if same_day_others.empty:
        return np.nan
    sims = [cosine_sim(row['embedding'], other_emb) for other_emb in same_day_others['embedding']]
    return np.mean(sims)

In [ ]:
df['rho_i'] = df.apply(lambda row: compute_rho_i(row, df), axis=1)
rho_m_per_date = df.groupby('Date')['rho_i'].mean().rename('rho_m')
df = df.merge(rho_m_per_date, on='Date', how='left')
df['rho_div'] = df['rho_m'] - df['rho_i']

In [ ]:
df

In [ ]:
df_reg = df[df['vote_dissent'].isin([0.0, 1.0])].copy()
df_reg

In [ ]:
df_reg['rho_div'].describe()

In [ ]:
df_reg['vote_dissent'].value_counts()

In [ ]:
# Définir les variables explicatives
X = df_reg[['rho_div', 'xp','Phd', 'gender', 'influence']]

# Ajouter l'intercept
X = sm.add_constant(X)

# Variable cible
y = df_reg['vote_dissent']

# Retirer les lignes avec valeurs manquantes
Xy = pd.concat([X, y], axis=1).dropna()
X = Xy.drop(columns='vote_dissent')
y = Xy['vote_dissent']

# Régression logistique
logit_model = sm.Logit(y, X).fit()
logit_model.summary()

In [ ]:
X = df_reg[['rho_div']]
X = sm.add_constant(X)
y = df_reg['vote_dissent']
logit_model = sm.Logit(y, X).fit()
logit_model.summary()